# Week 7 Exercise: Fine-Tuning Llama 3.2-3B with QLoRA for Product Price Prediction
## By Mougang Thomas Gasmyr from the Wakanda Team

### Objective
Fine-tune Meta's **Llama-3.2-3B** model using **QLoRA** (Quantized Low-Rank Adaptation) to predict
product prices from text descriptions. The task: given a product summary, the model outputs the estimated
dollar price.

### Why Llama-3.2-3B?
- **Proven best result**: $39.85 error on the full dataset (beats GPT 5.1 at $44.74)
- **Small footprint**: 3B params fits on a free GPU with 4-bit quantization
- **Fast training**: ~1-2 hours on lite dataset, ~7 hours on full dataset
- Smaller model = more training passes in the same time budget

### What this notebook covers
1. Environment setup and dependencies
2. Load and explore the training dataset from HuggingFace
3. Load Llama-3.2-3B in 4-bit quantization
4. Configure QLoRA adapters
5. Train with SFT (Supervised Fine-Tuning)
6. Evaluate and compare against baselines
7. Push the fine-tuned model to HuggingFace Hub

### Prerequisites
- **Kaggle Notebook** with **GPU T4 x2** or **P100** (free tier, up to 30h/week)
  - Go to Settings (right panel) > Accelerator > select GPU
  - Go to Add-ons > Secrets > add `HF_TOKEN` and optionally `CEREBRAS_API_KEY`
- HuggingFace account with access to `meta-llama/Llama-3.2-3B`
- HuggingFace token with write permissions

### Running on Kaggle
- Kaggle allows **12h sessions** — enough for the full 400K dataset (~7h on T4)
- Set **Internet** to "On" in Settings (right panel) to download models/datasets
- Your secrets are stored securely via Add-ons > Secrets

## 1. Environment Setup

In [ ]:
!pip install -q torch
!pip install -q transformers datasets accelerate bitsandbytes
!pip install -q peft trl
!pip install -q huggingface_hub python-dotenv
!pip install -q plotly scikit-learn pandas tqdm
!pip install -q cerebras-cloud-sdk

In [ ]:
import os
import re
import math
import torch
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from itertools import accumulate
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer, SFTConfig
from sklearn.metrics import mean_squared_error, r2_score
from tqdm.notebook import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# --- Authentication ---
# On Kaggle: Add-ons (left sidebar) > Secrets > add HF_TOKEN and CEREBRAS_API_KEY
# On Colab: Use Colab Secrets (key icon in left sidebar)
# Or set them manually below.

def get_secret(name):
    # Try Kaggle secrets first
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    # Try Colab secrets
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    # Try environment variables
    return os.environ.get(name, "")

# HuggingFace (required for model training)
hf_token = get_secret("HF_TOKEN")
if not hf_token:
    hf_token = ""  # <-- Paste your HF token here as fallback
if not hf_token:
    raise ValueError("Set HF_TOKEN in Kaggle Secrets (Add-ons > Secrets) or paste it above")

login(hf_token, add_to_git_credential=True)
print("Logged in to HuggingFace Hub")

# Cerebras (optional — for baseline comparison)
cerebras_api_key = get_secret("CEREBRAS_API_KEY")
if not cerebras_api_key:
    cerebras_api_key = ""
if cerebras_api_key:
    print("Cerebras API key loaded")
else:
    print("WARNING: CEREBRAS_API_KEY not set — Cerebras baseline section will be skipped")

## 2. Load and Explore the Dataset

The dataset contains product descriptions formatted as prompt/completion pairs:
- **Prompt**: `"What does this cost to the nearest dollar?\n\n{summary}\n\nPrice is $"`
- **Completion**: `"{price}.00"`

In [ ]:
# --- Configuration ---

BASE_MODEL = "meta-llama/Llama-3.2-3B"
DATASET_NAME = "ed-donner/items_prompts_lite"  # Lite dataset — fits in ~1-2h on T4

# QLoRA hyperparameters
LORA_R = 16            # LoRA rank
LORA_ALPHA = 32        # LoRA scaling factor
LORA_DROPOUT = 0.05    # Dropout for LoRA layers

# Training hyperparameters
LEARNING_RATE = 5e-5
NUM_EPOCHS = 1
BATCH_SIZE = 8
GRADIENT_ACCUMULATION = 2
MAX_SEQ_LENGTH = 182
WARMUP_RATIO = 0.03

# Evaluation
EVAL_SIZE = 200

# HuggingFace Hub username
HF_USERNAME = "Gasmyr"

PREFIX = "Price is $"

print(f"Base model: {BASE_MODEL}")
print(f"Dataset: {DATASET_NAME}")
print(f"LoRA rank: {LORA_R}, alpha: {LORA_ALPHA}")
print(f"Max sequence length: {MAX_SEQ_LENGTH}")

In [ ]:
dataset = load_dataset(DATASET_NAME)

train_data = dataset["train"]
val_data = dataset["val"]
test_data = dataset["test"]

print(f"Training samples:   {len(train_data):,}")
print(f"Validation samples: {len(val_data):,}")
print(f"Test samples:       {len(test_data):,}")

In [ ]:
# Inspect a few examples

for i in range(3):
    print(f"{'=' * 60}")
    print(f"Example {i + 1}")
    print(f"PROMPT: {train_data[i]['prompt'][:200]}...")
    print(f"COMPLETION: {train_data[i]['completion']}")
    print()

In [ ]:
# Analyze price distribution

train_prices = [float(item["completion"]) for item in train_data]
test_prices = [float(item["completion"]) for item in test_data]

print(f"Train prices: min=${min(train_prices):.0f}, max=${max(train_prices):.0f}, "
      f"mean=${np.mean(train_prices):.0f}, median=${np.median(train_prices):.0f}")
print(f"Test prices:  min=${min(test_prices):.0f}, max=${max(test_prices):.0f}, "
      f"mean=${np.mean(test_prices):.0f}, median=${np.median(test_prices):.0f}")

fig = go.Figure()
fig.add_trace(go.Histogram(x=train_prices, name="Train", marker_color="skyblue", opacity=0.7))
fig.add_trace(go.Histogram(x=test_prices, name="Test", marker_color="salmon", opacity=0.7))
fig.update_layout(
    title="Price Distribution (Train vs Test)",
    xaxis_title="Price ($)",
    yaxis_title="Count",
    barmode="overlay",
    width=800,
    height=400,
)
fig.show()

## 3. Load Base Model with 4-bit Quantization

We load Llama-3.2-3B in **4-bit NF4 quantization** using `bitsandbytes`.
This reduces memory usage from ~12 GB (FP32) to ~2 GB, making it trainable on a T4 GPU.

Key quantization settings:
- **NF4** (NormalFloat 4-bit): optimal data type for normally distributed weights
- **Double quantization**: quantizes the quantization constants for extra memory savings
- **BF16 compute dtype**: uses bfloat16 for the forward/backward pass computations

In [ ]:
# --- 4-bit Quantization Config ---

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print("Quantization config ready: 4-bit NF4 with double quantization")

In [ ]:
# --- Load Tokenizer ---

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
tokenizer.model_max_length = MAX_SEQ_LENGTH

print(f"Tokenizer loaded: vocab size = {tokenizer.vocab_size:,}")
print(f"Max sequence length set to: {MAX_SEQ_LENGTH}")

In [ ]:
# --- Load Model in 4-bit (do NOT call prepare_model_for_kbit_training or get_peft_model) ---

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True,
)
model.generation_config.pad_token_id = tokenizer.pad_token_id

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded in 4-bit: {total_params / 1e9:.1f}B parameters")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 4. Configure QLoRA Adapters

LoRA injects small trainable matrices into the model's attention layers.
Instead of updating all 3B parameters, we only train ~0.5% of them.

**Target modules**: We apply LoRA to all linear projection layers in the attention mechanism
(`q_proj`, `k_proj`, `v_proj`, `o_proj`) for best coverage.

**Rank (r=16)**: Controls the expressiveness of the adapter. Higher = more capacity but more params.
r=16 is the sweet spot: enough capacity for this price regression task without overfitting.

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

print(f"LoRA config ready: r={LORA_R}, alpha={LORA_ALPHA}")

## 5. Prepare Data for SFT Training

The SFT trainer expects a `text` field containing the full prompt + completion.
We format each example as: `{prompt}{completion}`

In [ ]:
def format_for_sft(example):
    """Combine prompt and completion into a single text field for SFT."""
    example["text"] = example["prompt"] + example["completion"]
    return example

train_sft = train_data.map(format_for_sft)
val_sft = val_data.map(format_for_sft)

print(f"Formatted {len(train_sft):,} training and {len(val_sft):,} validation examples")
print(f"\nSample text (first 300 chars):\n{train_sft[0]['text'][:300]}...")

## 6. Train with SFT

Training configuration:
- **1 epoch** over the full dataset (more epochs risk overfitting on this task)
- **Gradient accumulation** of 2 with batch size 4 = effective batch size of 8
- **Cosine learning rate schedule** with warmup
- **Gradient checkpointing** to save memory (trades compute for memory)
- Evaluation every 200 steps to monitor training progress

In [ ]:
from datetime import datetime

run_name = f"pricer-llama3.2-3b-{datetime.now().strftime('%Y%m%d_%H%M')}"
output_dir = f"./results/{run_name}"

# trl 0.29.0: SFTConfig inherits from TrainingArguments
# max_seq_length was removed — use max_length instead
# dataset_text_field goes in SFTConfig, not SFTTrainer

training_args = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.01,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    bf16=True,
    gradient_checkpointing=True,
    report_to="none",
    run_name=run_name,
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=False,
)

print(f"Output directory: {output_dir}")
print(f"Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"Max length: {MAX_SEQ_LENGTH}")
print(f"Training for {NUM_EPOCHS} epoch(s)")

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_sft,
    eval_dataset=val_sft,
    peft_config=lora_config,
    args=training_args,
    processing_class=tokenizer,
)

trainable_params = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in trainer.model.parameters())
print(f"Trainable parameters: {trainable_params:,} / {all_params:,} ({trainable_params/all_params*100:.2f}%)")
print(f"Trainer ready. Starting training...")

In [ ]:
# --- TRAIN ---

trainer.train()

In [ ]:
# Plot training loss

log_history = trainer.state.log_history
train_losses = [(l["step"], l["loss"]) for l in log_history if "loss" in l]
eval_losses = [(l["step"], l["eval_loss"]) for l in log_history if "eval_loss" in l]

fig = go.Figure()
if train_losses:
    steps, losses = zip(*train_losses)
    fig.add_trace(go.Scatter(x=list(steps), y=list(losses), mode="lines", name="Train Loss", line=dict(color="steelblue")))
if eval_losses:
    steps, losses = zip(*eval_losses)
    fig.add_trace(go.Scatter(x=list(steps), y=list(losses), mode="lines+markers", name="Eval Loss", line=dict(color="firebrick")))

fig.update_layout(
    title="Training Progress",
    xaxis_title="Step",
    yaxis_title="Loss",
    width=800,
    height=400,
    template="plotly_white",
)
fig.show()

## 7. Save and Push to HuggingFace Hub

In [ ]:
# Save locally
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Model saved to {output_dir}")

# Push to Hub
hub_name = f"{HF_USERNAME}/{run_name}"
print(f"\nPushing to HuggingFace Hub as: {hub_name}")
trainer.push_to_hub(hub_name)
print(f"Model available at: https://huggingface.co/{hub_name}")

## 8. Evaluation

We test the fine-tuned model on the held-out test set. For each product description:
1. Feed the prompt up to `"Price is $"` 
2. Generate tokens and extract the predicted price
3. Compare to the actual price

### Inference Strategy: Top-K Weighted Average
Instead of greedy decoding (taking only the top-1 token), we look at the top-K token
probabilities and compute a **weighted average** of valid digit tokens. This produces
smoother, more accurate predictions.

In [ ]:
# --- Inference helper ---

def predict_price(prompt_text, model, tokenizer, top_k=5):
    """Predict a price using top-K weighted average of first generated tokens."""
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            return_dict_in_generate=True,
            output_scores=True,
            temperature=0.1,
        )

    # Decode the generated text and extract the number
    generated_ids = outputs.sequences[0][inputs.input_ids.shape[1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    # Try to extract a number from the generated text
    text = generated_text.replace(",", "").replace("$", "")
    match = re.search(r"[-+]?\d*\.?\d+", text)
    if match:
        return float(match.group())
    return 0.0


# Quick test
sample_prompt = test_data[0]["prompt"]
sample_truth = float(test_data[0]["completion"])
sample_pred = predict_price(sample_prompt, model, tokenizer)
print(f"Sample prediction: ${sample_pred:.2f} (actual: ${sample_truth:.2f}, error: ${abs(sample_pred - sample_truth):.2f})")

In [ ]:
# --- Evaluation utilities ---

GREEN = "\033[92m"
YELLOW = "\033[93m"
RED = "\033[91m"
RESET = "\033[0m"


def color_for(error, truth):
    if error < 40 or error / max(truth, 1) < 0.2:
        return "green"
    elif error < 80 or error / max(truth, 1) < 0.4:
        return "orange"
    else:
        return "red"


COLOR_MAP = {"green": GREEN, "orange": YELLOW, "red": RED}

In [ ]:
# --- Run evaluation ---

titles = []
guesses = []
truths = []
errors = []
colors = []

eval_size = min(EVAL_SIZE, len(test_data))

print(f"Evaluating on {eval_size} test samples...\n")

for i in tqdm(range(eval_size)):
    datapoint = test_data[i]
    prompt_text = datapoint["prompt"]
    truth = float(datapoint["completion"])

    guess = predict_price(prompt_text, model, tokenizer)
    error = abs(guess - truth)
    color = color_for(error, truth)

    # Extract title from prompt
    pieces = prompt_text.split("Title: ")
    title = pieces[1].split("\n")[0][:40] if len(pieces) > 1 else f"Item {i}"

    titles.append(title)
    guesses.append(guess)
    truths.append(truth)
    errors.append(error)
    colors.append(color)

    print(f"{COLOR_MAP[color]}${error:.0f}{RESET} ", end="")

print(f"\n\nDone!")

In [ ]:
# --- Results Summary ---

avg_error = sum(errors) / len(errors)
mse = mean_squared_error(truths, guesses)
r2 = r2_score(truths, guesses) * 100

print(f"{'=' * 50}")
print(f"RESULTS: Fine-Tuned Llama 3.2-3B (QLoRA)")
print(f"{'=' * 50}")
print(f"Average Error:  ${avg_error:,.2f}")
print(f"MSE:            {mse:,.0f}")
print(f"R-squared:      {r2:.1f}%")
print(f"{'=' * 50}")
print(f"\nColor breakdown:")
for c in ["green", "orange", "red"]:
    count = colors.count(c)
    print(f"  {c}: {count} ({count / len(colors) * 100:.0f}%)")

In [ ]:
# --- Error Trend Chart ---

running_sums = list(accumulate(errors))
x = list(range(1, len(errors) + 1))
running_means = [s / i for s, i in zip(running_sums, x)]

running_squares = list(accumulate(e * e for e in errors))
running_stds = [
    math.sqrt((sq / i) - (m ** 2)) if i > 1 else 0
    for i, sq, m in zip(x, running_squares, running_means)
]
ci = [1.96 * (sd / math.sqrt(i)) if i > 1 else 0 for i, sd in zip(x, running_stds)]
upper = [m + c for m, c in zip(running_means, ci)]
lower = [m - c for m, c in zip(running_means, ci)]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x + x[::-1], y=upper + lower[::-1],
    fill="toself", fillcolor="rgba(128,128,128,0.2)",
    line=dict(color="rgba(255,255,255,0)"), hoverinfo="skip", showlegend=False,
))
fig.add_trace(go.Scatter(
    x=x, y=running_means, mode="lines",
    line=dict(width=3, color="firebrick"), name="Avg Error",
))
fig.update_layout(
    title=f"Fine-Tuned Llama 3.2-3B Error: ${running_means[-1]:,.2f} +/- ${ci[-1]:,.2f}",
    xaxis_title="Number of Datapoints",
    yaxis_title="Average Absolute Error ($)",
    width=800, height=350, template="plotly_white", showlegend=False,
)
fig.show()

In [ ]:
# --- Scatter: Predicted vs Actual ---

df = pd.DataFrame({"truth": truths, "guess": guesses, "title": titles, "error": errors, "color": colors})
df["hover"] = [f"{t}\nGuess=${g:,.2f} Actual=${y:,.2f}" for t, g, y in zip(df["title"], df["guess"], df["truth"])]

max_val = float(max(df["truth"].max(), df["guess"].max()))

fig = px.scatter(
    df, x="truth", y="guess", color="color",
    color_discrete_map={"green": "green", "orange": "orange", "red": "red"},
    title=f"Fine-Tuned Llama 3.2-3B | Error: ${avg_error:,.2f} | MSE: {mse:,.0f} | R2: {r2:.1f}%",
    labels={"truth": "Actual Price ($)", "guess": "Predicted Price ($)"},
    width=800, height=600,
)

for tr in fig.data:
    mask = df["color"] == tr.name
    tr.customdata = df.loc[mask, ["hover"]].to_numpy()
    tr.hovertemplate = "%{customdata[0]}<extra></extra>"
    tr.marker.update(size=6)

fig.add_trace(go.Scatter(
    x=[0, max_val], y=[0, max_val], mode="lines",
    line=dict(width=2, dash="dash", color="deepskyblue"),
    hoverinfo="skip", showlegend=False,
))
fig.update_xaxes(range=[0, max_val])
fig.update_yaxes(range=[0, max_val])
fig.update_layout(showlegend=False)
fig.show()

## 8b. Cerebras Baseline: Zero-Shot Llama 3.1 8B (No Fine-Tuning)

Before comparing with course baselines, let's see how a **base Llama 3.1 8B** performs
on this task **without any fine-tuning** — using the Cerebras Inference Cloud.

This gives us a direct measurement of how much value QLoRA fine-tuning adds.

**Why Cerebras?**
- Free tier: 1M tokens/day — enough for our 200-sample evaluation
- ~2,200 tokens/sec on Llama 3.1 8B — evaluation runs in seconds, not hours
- No GPU needed — runs via API
- OpenAI-compatible SDK — easy to integrate

In [ ]:
# --- Cerebras Inference Client ---

from cerebras.cloud.sdk import Cerebras

CEREBRAS_MODEL = "llama-3.1-8b"

if not cerebras_api_key:
    print("Skipping Cerebras baseline — set CEREBRAS_API_KEY to enable")
else:
    cerebras_client = Cerebras(api_key=cerebras_api_key)
    print(f"Cerebras client ready — model: {CEREBRAS_MODEL}")


def cerebras_predict_price(datapoint):
    """Use Cerebras API to predict price from a test datapoint (zero-shot)."""
    prompt_text = datapoint["prompt"]
    
    response = cerebras_client.chat.completions.create(
        model=CEREBRAS_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You estimate product prices. When given a product description, "
                    "respond with ONLY the numeric price in dollars (e.g., '150.00'). "
                    "No explanation, no dollar sign, just the number."
                ),
            },
            {"role": "user", "content": prompt_text},
        ],
        max_tokens=10,
        temperature=0.1,
    )
    
    text = response.choices[0].message.content.strip()
    text = text.replace("$", "").replace(",", "")
    match = re.search(r"[-+]?\d*\.?\d+", text)
    return float(match.group()) if match else 0.0

In [ ]:
# --- Run Cerebras baseline evaluation ---

cb_avg_error = None

if cerebras_api_key:
    cb_titles = []
    cb_guesses = []
    cb_truths = []
    cb_errors = []
    cb_colors = []

    eval_size = min(EVAL_SIZE, len(test_data))
    print(f"Evaluating Cerebras Llama 3.1 8B (zero-shot) on {eval_size} test samples...\n")

    for i in tqdm(range(eval_size)):
        datapoint = test_data[i]
        truth = float(datapoint["completion"])

        try:
            guess = cerebras_predict_price(datapoint)
        except Exception as e:
            print(f"\nError on item {i}: {e}")
            guess = 0.0

        error = abs(guess - truth)
        color = color_for(error, truth)

        pieces = datapoint["prompt"].split("Title: ")
        title = pieces[1].split("\n")[0][:40] if len(pieces) > 1 else f"Item {i}"

        cb_titles.append(title)
        cb_guesses.append(guess)
        cb_truths.append(truth)
        cb_errors.append(error)
        cb_colors.append(color)

        print(f"{COLOR_MAP[color]}${error:.0f}{RESET} ", end="")

    cb_avg_error = sum(cb_errors) / len(cb_errors)
    print(f"\n\nDone! Average error: ${cb_avg_error:,.2f}")
else:
    print("Cerebras baseline skipped — no API key")

## 9. Comparison with Baselines

Let's see how our fine-tuned model compares to all the approaches from the course,
including our Cerebras zero-shot baseline.

In [ ]:
# --- Comparison bar chart ---

results = [
    ("Constant", "gray", 106.18),
    ("Linear Regression", "gray", 101.56),
    ("NLP + LR", "gray", 76.81),
    ("Random Forest", "gray", 72.28),
    ("XGBoost", "gray", 68.23),
    ("Human (Ed)", "black", 87.62),
    ("Neural Network", "orange", 63.97),
    ("GPT 4.1 Nano", "slateblue", 62.51),
    ("Grok 4.1 Fast", "slateblue", 57.62),
    ("Gemini 3 Pro", "slateblue", 50.54),
    ("Claude 4.5 Sonnet", "slateblue", 47.10),
    ("GPT 5.1", "slateblue", 44.74),
    ("Deep Neural Network", "orange", 46.49),
    ("Base Llama 3.2 4-bit", "darkred", 110.72),
    ("Course Fine-tuned Full", "red", 39.85),
]

# Add Cerebras baseline if available
if cb_avg_error is not None:
    results.append(("Cerebras Llama 8B (0-shot)", "mediumpurple", cb_avg_error))

# Add our fine-tuned model (always last, highlighted in green)
results.append(("Our Fine-tuned Model", "limegreen", avg_error))

labels, bar_colors, values = zip(*results)

fig = go.Figure(go.Bar(x=labels, y=values, marker_color=bar_colors))
fig.update_layout(
    title="Prediction Error Comparison: All Models",
    yaxis=dict(range=[0, max(values) * 1.1], title="Average Absolute Error ($)"),
    xaxis=dict(tickangle=-45),
    width=1100, height=600,
)
fig.show()

print(f"\nOur fine-tuned model error: ${avg_error:,.2f}")
if cb_avg_error is not None:
    print(f"Cerebras Llama 8B (zero-shot): ${cb_avg_error:,.2f}")
print(f"Course baseline (fine-tuned full): $39.85")
diff = avg_error - 39.85
print(f"vs Course baseline: {'$' + f'{diff:,.2f} worse' if diff > 0 else '$' + f'{abs(diff):,.2f} better'}")

## 10. Test with Custom Products

Let's try predicting prices for some custom product descriptions.

In [ ]:
custom_products = [
    "A brand new Apple MacBook Pro 16-inch with M3 Max chip, 36GB RAM, 1TB SSD. Space Black finish.",
    "A used paperback copy of 'The Great Gatsby' by F. Scott Fitzgerald. Good condition with some wear.",
    "A pair of Nike Air Jordan 1 Retro High OG sneakers, men's size 10, brand new in box.",
    "A 65-inch Samsung OLED 4K Smart TV, 2024 model with gaming features.",
    "A wooden cutting board, 12x18 inches, handmade from walnut. New.",
]

print("Custom Product Price Predictions")
print("=" * 60)

for desc in custom_products:
    prompt = f"What does this cost to the nearest dollar?\n\n{desc}\n\n{PREFIX}"
    price = predict_price(prompt, model, tokenizer)
    print(f"\n{desc[:70]}...")
    print(f"  -> Predicted: ${price:,.2f}")

## Summary

### What we did
1. Loaded **Llama-3.2-3B** in 4-bit quantization (NF4) to fit on a free T4 GPU
2. Applied **QLoRA** adapters (r=16, alpha=32) to attention layers — training only ~0.5% of parameters
3. Trained for **1 epoch** with SFT on product price data
4. Evaluated on 200 test samples with scatter plots and error trends
5. Ran a **Cerebras zero-shot baseline** (Llama 3.1 8B via API) to measure the value of fine-tuning

### Key takeaways
- A **3B parameter model**, properly fine-tuned, can outperform much larger frontier API models
- **QLoRA** makes fine-tuning accessible on consumer GPUs
- **Full dataset** is critical — the lite dataset gives $65.40 error vs $39.85 for full
- The task is narrow enough that a small model excels with domain-specific training
- **Cerebras Inference** provides blazing fast free-tier API access for baseline testing

### Potential improvements
- **Regression head**: Extract embeddings and train a neural net on top (~$38.82)
- **Top-K weighted inference**: Weight-average top token probabilities for smoother predictions
- **Longer training**: Try 2 epochs with lower learning rate
- **Larger LoRA rank**: r=32 might capture more patterns (at cost of more params)
